# Comprehensive Audio Feature Extraction for Micro-Milling Process Analysis

This notebook implements extensive feature extraction from acoustic emission signals in micro-milling operations. The features are grounded in the machining acoustics and signal processing literature and are designed to capture variance in cutting depth and process conditions.

## Literature Foundation

The feature set is based on research in:
- Acoustic emission (AE) in machining (Teti et al., 2010; Sick, 2002)
- Tool condition monitoring via sound analysis (Alonso & Salgado, 2008)
- Surface roughness prediction from audio (Li & Yuan, 2005)
- Cutting force estimation from vibro-acoustic signals (Zhu et al., 2020)
- Time-frequency analysis for chatter detection (Yao et al., 2010)

## Feature Categories

1. **Time-Domain Features**: Amplitude statistics, envelope characteristics, temporal structure
2. **Frequency-Domain Features**: Spectral shape, energy distribution, harmonic content
3. **Statistical Features**: Higher-order moments, distribution characteristics
4. **Energy-Based Features**: Band power ratios, frequency-localized energy
5. **Machining-Specific Features**: Tool chatter indicators, cutting force proxies, surface roughness correlates
6. **Time-Frequency Features**: Spectral dynamics, modulation characteristics

In [ ]:
# Core imports
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import soundfile as sf
import scipy.signal as sig
import scipy.stats as stats
from scipy.fft import fft, fftfreq
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

## 1. Audio Loading and Preprocessing

We reuse the essential loading functions from the main analysis, ensuring consistency in how we handle the audio files and extract stable segments for feature computation.

In [ ]:
# File paths - update these to your actual files
FILE_PATHS = [
    "4x0,1mm_Nut_keineLuft_oben_front_cut.flac",
    "4x0,2mm_Nut_keineLuft_oben_front_cut.flac",
    "4x0,3mm_Nut_keineLuft_oben_front_cut.flac",
    "4x0,4mm_Nut_keineLuft_oben_front_cut.flac",
    "4x0,5mm_Nut_keineLuft_oben_front_cut.flac"
]

TARGET_SR = 192000  # High sample rate typical for acoustic emission monitoring
SKIP_START_S = 1.33  # Skip startup transients
SKIP_END_S = 0.5     # Skip end transients

In [ ]:
def parse_depth_mm(name: str) -> float:
    """Extract cutting depth from filename.
    
    Handles formats like '4x0,2mm' or '4x0.2mm' where depth is after 'x'.
    """
    m = re.search(r"x\s*(\d+)\s*([,.])\s*(\d+)\s*mm", name, flags=re.IGNORECASE)
    if m:
        return float(f"{m.group(1)}.{m.group(3)}")
    
    m = re.search(r"(\d+)\s*([,.])\s*(\d+)\s*mm", name, flags=re.IGNORECASE)
    if m:
        return float(f"{m.group(1)}.{m.group(3)}")
    
    return float("nan")


def load_mono(
    path: str,
    target_sr: int | None = None,
    *,
    allow_resample: bool = True,
    normalize: bool = True,
) -> tuple[np.ndarray, int]:
    """Load audio file as mono, optionally resample and normalize.
    
    Parameters:
    -----------
    path : str
        Path to audio file
    target_sr : int, optional
        Target sample rate for resampling
    allow_resample : bool
        Whether to allow resampling if sample rates don't match
    normalize : bool
        Whether to normalize to [-1, 1] range
    
    Returns:
    --------
    y : ndarray
        Audio signal
    sr : int
        Sample rate
    """
    y, sr = sf.read(path, always_2d=False)

    # Convert to mono if stereo
    if y.ndim > 1:
        y = y.mean(axis=1)

    # Resample if needed
    if target_sr and sr != target_sr:
        if not allow_resample:
            raise ValueError(f"Sample rate mismatch: {sr} != {target_sr}")
        # Simple resampling using scipy
        num_samples = int(len(y) * target_sr / sr)
        y = sig.resample(y, num_samples)
        sr = target_sr

    # Normalize
    if normalize:
        peak = np.max(np.abs(y))
        if peak > 1e-8:
            y = y / peak

    return y, sr

In [ ]:
def extract_stable_segment(y: np.ndarray, sr: int, 
                          skip_start_s: float = SKIP_START_S,
                          skip_end_s: float = SKIP_END_S) -> np.ndarray:
    """Extract stable cutting segment by removing startup and ending transients.
    
    This is critical for machining audio analysis as startup and shutdown
    phases have very different acoustic characteristics than steady-state cutting.
    """
    i_start = int(skip_start_s * sr)
    i_end = len(y) - int(skip_end_s * sr)
    return y[i_start:i_end]

In [ ]:
# Load all recordings
records = []

for path in FILE_PATHS:
    try:
        y, sr = load_mono(path, target_sr=TARGET_SR)
        seg = extract_stable_segment(y, sr)
        
        records.append({
            "name": Path(path).name,
            "depth_mm": parse_depth_mm(Path(path).name),
            "y": y,
            "seg": seg,
            "sr": sr
        })
        print(f"Loaded {Path(path).name}: depth={parse_depth_mm(Path(path).name):.1f}mm, "
              f"duration={len(seg)/sr:.2f}s")
    except FileNotFoundError:
        print(f"Warning: File not found: {path}")

print(f"\nSuccessfully loaded {len(records)} recordings")

## 2. Time-Domain Features

Time-domain features capture the amplitude characteristics and temporal structure of the acoustic emission signal. In machining, these features correlate with cutting forces, tool-workpiece contact dynamics, and chip formation processes.

### Theoretical Background

**RMS (Root Mean Square)**: Directly related to signal energy and cutting forces. Higher RMS typically indicates more aggressive cutting or greater material removal (Li & Yuan, 2005).

**Crest Factor**: Ratio of peak to RMS, indicating the "impulsiveness" of the signal. In machining, high crest factors can indicate intermittent cutting, tool wear, or chatter (Sick, 2002).

**Kurtosis**: Measures "tailedness" of amplitude distribution. Values >3 indicate impulsive behavior; in machining this can relate to tool chatter or defects (Teti et al., 2010).

**Zero Crossing Rate (ZCR)**: Frequency content proxy. In milling, ZCR relates to tooth passing frequency and can indicate surface finish quality.

In [ ]:
def compute_time_domain_features(x: np.ndarray, sr: int) -> Dict[str, float]:
    """
    Compute comprehensive time-domain features from audio signal.
    
    Parameters:
    -----------
    x : ndarray
        Audio signal (mono)
    sr : int
        Sample rate
    
    Returns:
    --------
    features : dict
        Dictionary of feature names and values
    """
    features = {}
    
    # Basic amplitude statistics
    features['rms'] = float(np.sqrt(np.mean(x**2)))
    features['peak_amplitude'] = float(np.max(np.abs(x)))
    features['mean_amplitude'] = float(np.mean(np.abs(x)))
    
    # Crest factor: indicates signal impulsiveness
    # High values suggest transient/impulsive events (tool chatter, impacts)
    features['crest_factor'] = features['peak_amplitude'] / (features['rms'] + 1e-10)
    
    # Shape factor: another impulsiveness measure
    features['shape_factor'] = features['rms'] / (features['mean_amplitude'] + 1e-10)
    
    # Impulse factor: combines peak and mean
    features['impulse_factor'] = features['peak_amplitude'] / (features['mean_amplitude'] + 1e-10)
    
    # Clearance factor: sensitive to peak values
    square_root_amplitude = np.mean(np.sqrt(np.abs(x)))**2
    features['clearance_factor'] = features['peak_amplitude'] / (square_root_amplitude + 1e-10)
    
    # Higher-order statistics
    features['kurtosis'] = float(stats.kurtosis(x, fisher=True))  # Excess kurtosis
    features['skewness'] = float(stats.skew(x))


    # Teager-Kaiser Energy Operator (TKEO): instantaneous energy / impacts
    # Common in acoustic/vibration monitoring for impulsiveness and chatter-related bursts
    if len(x) >= 3:
        tkeo = x[1:-1]**2 - x[:-2]*x[2:]
        tkeo = np.abs(tkeo)
        features['tkeo_mean'] = float(np.mean(tkeo))
        features['tkeo_std'] = float(np.std(tkeo))
    else:
        features['tkeo_mean'] = 0.0
        features['tkeo_std'] = 0.0    
    # Energy and power
    features['energy'] = float(np.sum(x**2))
    features['log_energy'] = float(np.log(features['energy'] + 1e-10))
    
    # Zero crossing rate: frequency content indicator
    # High ZCR suggests high-frequency content, potentially from tool vibration
    zero_crossings = np.sum(np.diff(np.sign(x)) != 0)
    features['zcr'] = zero_crossings / (2 * len(x))
    features['zcr_hz'] = features['zcr'] * sr  # Convert to Hz
    
    # Envelope features (using Hilbert transform)
    envelope = np.abs(sig.hilbert(x))
    features['envelope_mean'] = float(np.mean(envelope))
    features['envelope_std'] = float(np.std(envelope))
    features['envelope_max'] = float(np.max(envelope))
    
    # Envelope dynamic range
    features['envelope_dynamic_range_db'] = 20 * np.log10(
        features['envelope_max'] / (np.percentile(envelope, 10) + 1e-10)
    )
    
    # Waveform length: total variation in signal
    # Sensitive to signal complexity and roughness
    features['waveform_length'] = float(np.sum(np.abs(np.diff(x))))
    
    # Percentile-based features
    abs_x = np.abs(x)
    for p in [10, 25, 50, 75, 90, 95, 99]:
        features[f'percentile_{p}'] = float(np.percentile(abs_x, p))
    
    # Inter-quartile range: measure of spread
    features['iqr'] = features['percentile_75'] - features['percentile_25']
    
    # Dynamic range in dB
    features['dynamic_range_db'] = 20 * np.log10(
        features['percentile_99'] / (features['percentile_10'] + 1e-10)
    )
    
    return features

## 3. Frequency-Domain Features

Frequency-domain features reveal the spectral content of machining sounds. Different cutting depths affect the frequency distribution due to changes in cutting forces, tool deflection, and workpiece resonances.

### Theoretical Background

**Spectral Centroid**: The "center of mass" of the spectrum. In machining, this shifts with cutting speed and depth. Deeper cuts often shift energy to lower frequencies due to increased structural vibrations (Alonso & Salgado, 2008).

**Spectral Bandwidth**: Spread of frequency content. Broader bandwidth can indicate more chaotic cutting dynamics or chatter.

**Spectral Rolloff**: Frequency below which 85% of spectral energy lies. Indicates the "brightness" of the sound.

**Spectral Entropy**: Disorder in frequency distribution. Higher entropy suggests more uniform spectral content (white noise-like), while lower entropy indicates tonal components.

In [ ]:
def compute_frequency_domain_features(x: np.ndarray, sr: int, 
                                     nfft: int = 8192) -> Dict[str, float]:
    """
    Compute comprehensive frequency-domain features.
    
    Parameters:
    -----------
    x : ndarray
        Audio signal
    sr : int
        Sample rate  
    nfft : int
        FFT size (zero-padded if needed for better frequency resolution)
    
    Returns:
    --------
    features : dict
        Dictionary of frequency-domain features
    """
    features = {}
    
    # Compute magnitude spectrum
    X = fft(x, n=nfft)
    freqs = fftfreq(nfft, 1/sr)
    
    # Only keep positive frequencies
    pos_mask = freqs >= 0
    freqs = freqs[pos_mask]
    mag = np.abs(X[pos_mask])
    
    # Power spectral density
    psd = mag**2 / len(x)
    psd_normalized = psd / (np.sum(psd) + 1e-10)
    
    # Spectral centroid: "center of gravity" of spectrum
    # Shifts with cutting conditions; deeper cuts often shift to lower frequencies
    features['spectral_centroid'] = float(np.sum(freqs * psd_normalized))
    
    # Spectral spread (standard deviation around centroid)
    features['spectral_spread'] = float(np.sqrt(
        np.sum(((freqs - features['spectral_centroid'])**2) * psd_normalized)
    ))
    
    # Spectral bandwidth (alternative measure)
    features['spectral_bandwidth'] = features['spectral_spread']
    
    # Spectral rolloff: frequency below which 85% of energy is contained
    cumsum_psd = np.cumsum(psd_normalized)
    rolloff_idx = np.where(cumsum_psd >= 0.85)[0]
    features['spectral_rolloff_85'] = float(freqs[rolloff_idx[0]] if len(rolloff_idx) > 0 else freqs[-1])
    
    # Also compute 95% rolloff
    rolloff_idx_95 = np.where(cumsum_psd >= 0.95)[0]
    features['spectral_rolloff_95'] = float(freqs[rolloff_idx_95[0]] if len(rolloff_idx_95) > 0 else freqs[-1])
    
    # Spectral skewness: asymmetry of spectrum
    features['spectral_skewness'] = float(stats.skew(psd_normalized))
    
    # Spectral kurtosis: "peakedness" of spectrum
    features['spectral_kurtosis'] = float(stats.kurtosis(psd_normalized, fisher=True))
    
    # Spectral flatness: measure of tonality vs. noise-like character
    # Values near 1 = noise-like, near 0 = tonal
    geometric_mean = np.exp(np.mean(np.log(psd + 1e-10)))
    arithmetic_mean = np.mean(psd)
    features['spectral_flatness'] = geometric_mean / (arithmetic_mean + 1e-10)
    
    # Spectral entropy: disorder in frequency distribution
    # Normalized to [0, 1] where 1 is maximum disorder
    psd_norm_entropy = psd_normalized + 1e-10
    features['spectral_entropy'] = float(-np.sum(psd_norm_entropy * np.log2(psd_norm_entropy)) / 
                                        np.log2(len(psd_norm_entropy)))
    
    # Spectral crest: ratio of peak to mean
    features['spectral_crest'] = float(np.max(psd) / (np.mean(psd) + 1e-10))
    
    # Spectral slope: overall trend of spectrum (linear regression)
    # Negative slope typical in natural sounds
    freq_log = np.log(freqs[1:] + 1)  # Avoid log(0)
    psd_db = 10 * np.log10(psd[1:] + 1e-10)
    coeffs = np.polyfit(freq_log, psd_db, 1)
    features['spectral_slope'] = float(coeffs[0])
    features['spectral_intercept'] = float(coeffs[1])
    
    # Peak frequencies: identify dominant frequency components
    # These often relate to tooth passing frequency in milling
    peak_indices = sig.find_peaks(psd, height=np.max(psd) * 0.1)[0]
    if len(peak_indices) > 0:
        peak_freqs = freqs[peak_indices]
        peak_mags = psd[peak_indices]
        sorted_idx = np.argsort(peak_mags)[::-1]  # Sort by magnitude
        
        # Store top 5 peak frequencies and their magnitudes
        for i in range(min(5, len(sorted_idx))):
            idx = sorted_idx[i]
            features[f'peak_freq_{i+1}'] = float(peak_freqs[idx])
            features[f'peak_mag_{i+1}'] = float(peak_mags[idx])
    else:
        for i in range(5):
            features[f'peak_freq_{i+1}'] = 0.0
            features[f'peak_mag_{i+1}'] = 0.0
    
    # Spectral decrease: how quickly spectrum decreases
    if len(psd) > 1:
        k = np.arange(1, len(psd))
        features['spectral_decrease'] = float(np.sum((psd[1:] - psd[0]) / k) / 
                                             (np.sum(psd[1:]) + 1e-10))
    else:
        features['spectral_decrease'] = 0.0
    
    return features

## 4. Band Power Features

Energy distribution across frequency bands reveals specific aspects of the machining process. Different frequency ranges carry different information about tool-workpiece interaction.

### Frequency Band Definitions (based on Teti et al., 2010)

- **Low band (100-1000 Hz)**: Structural vibrations, machine compliance, low-frequency chatter
- **Mid band (1-5 kHz)**: Cutting forces, chip formation, primary machining dynamics  
- **High band (5-20 kHz)**: Tool edge effects, surface generation, micro-scale phenomena
- **Very high band (20-80 kHz)**: Acoustic emission from crack formation, friction, wear

Band power ratios help normalize for overall amplitude differences and highlight spectral shape changes.

In [ ]:
def compute_band_power_features(x: np.ndarray, sr: int) -> Dict[str, float]:
    """
    Compute power in different frequency bands.
    
    Frequency bands are chosen based on machining acoustics literature:
    - Low frequencies: structural dynamics, large-scale vibrations
    - Mid frequencies: cutting forces, chip formation
    - High frequencies: tool edge, surface roughness
    - Very high frequencies: acoustic emission from material processes
    
    Parameters:
    -----------
    x : ndarray
        Audio signal
    sr : int
        Sample rate
    
    Returns:
    --------
    features : dict
        Band power features
    """
    features = {}
    
    # Define frequency bands (Hz)
    bands = {
        'very_low': (100, 2500),
        'low': (2500, 7500),
        'mid': (7500, 10000),
        'high': (10000, 12500),
        'very_high': (12500, 90000)
    }
    
    # Also define narrower bands for more detailed analysis
    detailed_bands = {
        'sub_500': (100, 500),
        'band_500_1.5k': (500, 1500),
        'band_1.5k_2.5k': (1500, 2500),
        'band_2.5k_5k': (2500, 5000),
        'band_5k_7.5k': (5000, 7500),
        'band_12.5k_20k': (12500, 20000),
        'band_20k_32k': (20000, 32000)
    }
    
    bands.update(detailed_bands)
    
    # Compute power in each band using Butterworth bandpass filter
    band_powers = {}
    for band_name, (f_low, f_high) in bands.items():
        # Skip bands outside Nyquist
        if f_low >= sr/2:
            band_powers[band_name] = 0.0
            continue
        
        # Adjust high frequency if needed
        f_high = min(f_high, sr/2 - 100)
        
        try:
            # Design bandpass filter
            sos = sig.butter(4, [f_low, f_high], btype='bandpass', 
                           fs=sr, output='sos')
            x_filtered = sig.sosfilt(sos, x)
            
            # RMS power in this band
            power = float(np.sqrt(np.mean(x_filtered**2)))
            band_powers[band_name] = power
            features[f'band_power_{band_name}'] = power
            features[f'band_power_{band_name}_db'] = 20 * np.log10(power + 1e-10)
        except ValueError:
            band_powers[band_name] = 0.0
            features[f'band_power_{band_name}'] = 0.0
            features[f'band_power_{band_name}_db'] = -100.0
    
    # Compute band power ratios (normalized features)
    total_power = sum(band_powers.values()) + 1e-10
    
    for band_name, power in band_powers.items():
        features[f'band_ratio_{band_name}'] = power / total_power
    
    # Specific ratios of interest in machining
    # Mid/Very low ratio: indicates balance between cutting forces and basic structural vibration
    features['ratio_mid_to_verylow'] = (band_powers.get('mid', 0) /
                                        (band_powers.get('very_low', 0) + 1e-10))
    
    # High/Very low ratio: indicates balance between surface effects and basic structural vibration
    features['ratio_high_to_verylow'] = (band_powers.get('high', 0) /
                                        (band_powers.get('very_low', 0) + 1e-10))
    
    # High/Low ratio: indicates balance between surface effects and structural vibration
    features['ratio_high_to_low'] = (band_powers.get('high', 0) / 
                                     (band_powers.get('low', 0) + 1e-10))
    
    # Mid/Low ratio: cutting forces vs structural dynamics
    features['ratio_mid_to_low'] = (band_powers.get('mid', 0) / 
                                   (band_powers.get('low', 0) + 1e-10))
    
    # Very high/Mid ratio: acoustic emission vs cutting forces
    features['ratio_veryhigh_to_mid'] = (band_powers.get('very_high', 0) / 
                                        (band_powers.get('mid', 0) + 1e-10))
    
    return features

## 5. Machining-Specific Features

These features are specifically designed based on the physics of machining processes and have been validated in the literature for tool condition monitoring, surface roughness prediction, and process monitoring.

### Theoretical Background

**Tool Chatter Indicators**: Chatter is self-excited vibration that degrades surface quality. It manifests as periodic amplitude modulation and specific frequency components (Yao et al., 2010).

**Cutting Force Proxies**: While we measure sound, many acoustic features correlate with cutting forces. RMS in mid-frequency bands is particularly well-correlated (Zhu et al., 2020).

**Surface Roughness Correlates**: High-frequency content (5-20 kHz) correlates with surface texture. Spectral characteristics in this band predict Ra values (Li & Yuan, 2005).

In [ ]:
def compute_machining_specific_features(x: np.ndarray, sr: int) -> Dict[str, float]:
    """
    Compute features specific to machining process monitoring.
    
    These features are grounded in machining acoustics research and correlate
    with process parameters like cutting forces, tool wear, chatter, and 
    surface roughness.
    
    Parameters:
    -----------
    x : ndarray
        Audio signal
    sr : int
        Sample rate
    
    Returns:
    --------
    features : dict
        Machining-specific features
    """
    features = {}
    
    # 1. Chatter detection features
    # Chatter manifests as amplitude modulation, detected via envelope analysis
    
    # Get analytic signal and envelope
    analytic = sig.hilbert(x)
    envelope = np.abs(analytic)
    
    # Detrend envelope to remove slow variations
    envelope_detrended = sig.detrend(envelope)
    
    # Envelope spectrum (FFT of envelope)
    env_fft = np.abs(fft(envelope_detrended, n=8192))
    env_freqs = fftfreq(8192, 1/sr)[:len(env_fft)//2]
    env_mag = env_fft[:len(env_fft)//2]

    # Temporal Peak Frequency (TPF): dominant modulation frequency of the envelope
    # Often aligns with tooth passing or unstable amplitude modulation components
    if len(env_freqs) > 0:
        f_hi = min(5000.0, float(env_freqs[-1]))
        band = (env_freqs >= 50.0) & (env_freqs <= f_hi)
        if np.any(band):
            idx = int(np.argmax(env_mag[band]))
            features['tpf_hz'] = float(env_freqs[band][idx])
        else:
            features['tpf_hz'] = 0.0
    else:
        features['tpf_hz'] = 0.0
        
    # Chatter frequency typically in 100-2500 Hz range for micro-milling
    chatter_band = (env_freqs >= 100) & (env_freqs <= 2500)
    features['chatter_indicator_0.1k_2.5kHz'] = float(np.sum(env_mag[chatter_band]))
    
    # Modulation depth: variation in envelope
    features['modulation_depth_0.1k_2.5kHz'] = float(np.std(envelope) / (np.mean(envelope) + 1e-10))
    
    # Envelope kurtosis: impulsiveness in amplitude envelope
    features['envelope_kurtosis_0.1k_2.5kHz'] = float(stats.kurtosis(envelope, fisher=True))
    
    # 2. Cutting force proxy features
    # Mid-frequency RMS correlates well with cutting forces
    
    # Filter to cutting force band (1-5 kHz)
    if sr > 10000:
        sos = sig.butter(4, [1000, 5000], btype='bandpass', fs=sr, output='sos')
        x_force_band = sig.sosfilt(sos, x)
        features['force_proxy_rms_1k_5kHz'] = float(np.sqrt(np.mean(x_force_band**2)))
        features['force_proxy_peak_1k_5kHz'] = float(np.max(np.abs(x_force_band)))
        
        # Moving RMS to capture force variations
        window_samples = int(0.01 * sr)  # 10ms window
        moving_rms = np.sqrt(sig.convolve(x_force_band**2, 
                                         np.ones(window_samples)/window_samples, 
                                         mode='same'))
        features['force_proxy_std_1k_5kHz'] = float(np.std(moving_rms))
        features['force_proxy_variation_1k_5kHz'] = features['force_proxy_std_1k_5kHz'] / (np.mean(moving_rms) + 1e-10)
    else:
        features['force_proxy_rms'] = 0.0
        features['force_proxy_peak'] = 0.0
        features['force_proxy_std'] = 0.0
        features['force_proxy_variation'] = 0.0
    
    # 3. Surface roughness correlates
    # High-frequency content (5-20 kHz) correlates with surface texture
    
    if sr > 40000:
        sos = sig.butter(4, [5000, min(12500, sr/2-100)], btype='bandpass', 
                        fs=sr, output='sos')
        x_roughness_band = sig.sosfilt(sos, x)
        features['roughness_proxy_rms_5k_12.5kHz'] = float(np.sqrt(np.mean(x_roughness_band**2)))
        
        # Spectral features in roughness band
        X_rough = fft(x_roughness_band, n=4096)
        freqs_rough = fftfreq(4096, 1/sr)[:2048]
        psd_rough = np.abs(X_rough[:2048])**2
        psd_rough_norm = psd_rough / (np.sum(psd_rough) + 1e-10)
        
        # Centroid in roughness band
        features['roughness_spectral_centroid'] = float(np.sum(freqs_rough * psd_rough_norm))
        
        # Entropy in roughness band
        psd_rough_norm_safe = psd_rough_norm + 1e-10
        features['roughness_spectral_entropy'] = float(
            -np.sum(psd_rough_norm_safe * np.log2(psd_rough_norm_safe)) / 
            np.log2(len(psd_rough_norm_safe))
        )
    else:
        features['roughness_proxy_rms'] = 0.0
        features['roughness_spectral_centroid'] = 0.0
        features['roughness_spectral_entropy'] = 0.0

    # 4. Tool wear indicators
    # Tool wear increases friction, shifting energy to higher frequencies
    # and increasing overall amplitude in acoustic emission band
    
    if sr > 40000:
        # Acoustic emission band (12.5-30 kHz)
        f_high_ae = min(30000, sr/2 - 100)
        if f_high_ae > 12500:
            sos = sig.butter(4, [12500, f_high_ae], btype='bandpass', 
                           fs=sr, output='sos')
            x_ae = sig.sosfilt(sos, x)
            features['acoustic_emission_rms'] = float(np.sqrt(np.mean(x_ae**2)))
            features['acoustic_emission_energy'] = float(np.sum(x_ae**2))
        else:
            features['acoustic_emission_rms'] = 0.0
            features['acoustic_emission_energy'] = 0.0
    else:
        features['acoustic_emission_rms'] = 0.0
        features['acoustic_emission_energy'] = 0.0
    
    # 5. Stability indicators
    # Process stability reflected in temporal consistency
    
    # Split signal into segments and compute RMS variation
    n_segments = 10
    segment_length = len(x) // n_segments
    segment_rms = []
    
    for i in range(n_segments):
        start = i * segment_length
        end = start + segment_length
        segment_rms.append(np.sqrt(np.mean(x[start:end]**2)))
    
    segment_rms = np.array(segment_rms)
    features['temporal_stability'] = float(np.std(segment_rms) / (np.mean(segment_rms) + 1e-10))
    features['temporal_trend'] = float(np.polyfit(np.arange(n_segments), segment_rms, 1)[0])
    
    # Autocorrelation at tool passing period (if known)
    # For now, compute autocorrelation decay as a general stability measure
    autocorr = np.correlate(x, x, mode='full')[len(x)-1:]
    autocorr = autocorr / autocorr[0]  # Normalize
    
    # Measure decay: how quickly autocorrelation decreases
    # Slower decay indicates more periodic/stable process
    lag_samples = min(int(0.01 * sr), len(autocorr) - 1)  # 10ms lag
    features['autocorr_10ms'] = float(autocorr[lag_samples])
    
    return features

## 6. Time-Frequency Features

Time-frequency features capture how spectral content changes over time. This is crucial for machining where the cutting process is inherently dynamic.

### Theoretical Background

**Spectral Flux**: Measures rate of change in spectrum. High flux indicates rapidly changing spectral content, possibly due to unstable cutting or chatter (Alonso & Salgado, 2008).

**Spectral Variation**: Variability in spectral shape over time. More stable processes show lower spectral variation.

**Frequency Modulation**: Changes in dominant frequencies can indicate tool deflection, workpiece vibration, or cutting parameter variations.

In [ ]:
def compute_timefrequency_features(x: np.ndarray, sr: int,
                                  nperseg: int = 8192,
                                  hop_length: int = 2048) -> Dict[str, float]:
    """
    Compute time-frequency domain features using STFT.
    
    These features capture spectral dynamics over time, important for
    detecting instabilities, chatter, and cutting parameter variations.
    
    Parameters:
    -----------
    x : ndarray
        Audio signal
    sr : int
        Sample rate
    nperseg : int
        STFT window size
    hop_length : int
        STFT hop size
    
    Returns:
    --------
    features : dict
        Time-frequency features
    """
    features = {}
    
    # Compute STFT
    freqs, times, Zxx = sig.stft(x, fs=sr, nperseg=nperseg, 
                                 noverlap=nperseg-hop_length)
    
    # Magnitude spectrogram
    mag_spec = np.abs(Zxx)
    power_spec = mag_spec ** 2


    # Dominant frequency over time (argmax per frame)
    if power_spec.shape[1] > 0:
        dom_idx = np.argmax(power_spec, axis=0)
        dom_freq = freqs[dom_idx]
        features['spec_dom_freq_hz'] = float(np.mean(dom_freq))
        features['spec_dom_freq_std_hz'] = float(np.std(dom_freq))
    else:
        features['spec_dom_freq_hz'] = 0.0
        features['spec_dom_freq_std_hz'] = 0.0

    # Tonalness over time (1 - spectral flatness)
    # Flatness ~1 noise-like, ~0 tonal; tonalness in [~0, 1]
    if power_spec.shape[1] > 0:
        gm = np.exp(np.mean(np.log(power_spec + 1e-10), axis=0))
        am = np.mean(power_spec + 1e-10, axis=0)
        flat = gm / (am + 1e-10)
        tonal = 1.0 - flat
        features['spec_tonalness_mean'] = float(np.mean(tonal))
    else:
        features['spec_tonalness_mean'] = 0.0    
    # 1. Spectral flux: measure of spectral change over time
    # Computed as Euclidean distance between successive spectra
    spectral_flux = []
    for i in range(1, mag_spec.shape[1]):
        flux = np.sqrt(np.sum((mag_spec[:, i] - mag_spec[:, i-1])**2))
        spectral_flux.append(flux)
    
    spectral_flux = np.array(spectral_flux)
    features['spectral_flux_mean'] = float(np.mean(spectral_flux))
    features['spectral_flux_std'] = float(np.std(spectral_flux))
    features['spectral_flux_max'] = float(np.max(spectral_flux))
    
    # 2. Spectral variation: overall change in spectral shape
    # Coefficient of variation for each frequency bin across time
    spectral_variation = np.std(mag_spec, axis=1) / (np.mean(mag_spec, axis=1) + 1e-10)
    features['spectral_variation_mean'] = float(np.mean(spectral_variation))
    features['spectral_variation_median'] = float(np.median(spectral_variation))
    
    # 3. Temporal centroid: when in time is energy concentrated
    energy_envelope = np.sum(power_spec, axis=0)
    energy_envelope_norm = energy_envelope / (np.sum(energy_envelope) + 1e-10)
    features['temporal_centroid'] = float(np.sum(times * energy_envelope_norm))
    
    # 4. Frequency band modulation
    # Track how energy in different bands varies over time
    
    # Define bands of interest
    bands_hz = {
        'very_low': (100, 2500),
        'low': (2500, 7500),
        'mid': (7500, 10000),
        'high': (10000, 12500),
        'very_high': (12500, 90000)
    }
    
    for band_name, (f_low, f_high) in bands_hz.items():
        # Find frequency bins in this band
        band_mask = (freqs >= f_low) & (freqs <= f_high)
        
        if np.any(band_mask):
            # Energy in this band over time
            band_energy_time = np.sum(power_spec[band_mask, :], axis=0)
            
            # Statistics of temporal variation
            features[f'{band_name}_band_energy_mean'] = float(np.mean(band_energy_time))
            features[f'{band_name}_band_energy_std'] = float(np.std(band_energy_time))
            features[f'{band_name}_band_energy_cv'] = float(
                np.std(band_energy_time) / (np.mean(band_energy_time) + 1e-10)
            )
    
    # 5. Spectral centroid modulation
    # Track how spectral centroid changes over time
    power_spec_norm = power_spec / (np.sum(power_spec, axis=0, keepdims=True) + 1e-10)
    centroid_over_time = np.sum(freqs[:, np.newaxis] * power_spec_norm, axis=0)
    
    features['centroid_modulation_mean'] = float(np.mean(centroid_over_time))
    features['centroid_modulation_std'] = float(np.std(centroid_over_time))
    features['centroid_modulation_range'] = float(np.max(centroid_over_time) - 
                                                  np.min(centroid_over_time))
    
    # 6. Bandwidth modulation
    bandwidth_over_time = np.sqrt(np.sum(
        ((freqs[:, np.newaxis] - centroid_over_time[np.newaxis, :])**2) * power_spec_norm,
        axis=0
    ))
    
    features['bandwidth_modulation_mean'] = float(np.mean(bandwidth_over_time))
    features['bandwidth_modulation_std'] = float(np.std(bandwidth_over_time))
    
    # 7. Spectral rolloff modulation
    rolloff_over_time = []
    for t in range(power_spec.shape[1]):
        cumsum = np.cumsum(power_spec[:, t])
        total = cumsum[-1]
        rolloff_idx = np.where(cumsum >= 0.85 * total)[0]
        if len(rolloff_idx) > 0:
            rolloff_over_time.append(freqs[rolloff_idx[0]])
    
    if len(rolloff_over_time) > 0:
        rolloff_over_time = np.array(rolloff_over_time)
        features['rolloff_modulation_mean'] = float(np.mean(rolloff_over_time))
        features['rolloff_modulation_std'] = float(np.std(rolloff_over_time))
    else:
        features['rolloff_modulation_mean'] = 0.0
        features['rolloff_modulation_std'] = 0.0
    
    return features

## 7. Statistical Summary Features

Additional statistical characterizations that complement the domain-specific features.

In [ ]:
def compute_statistical_features(x: np.ndarray) -> Dict[str, float]:
    """
    Compute general statistical features from signal.
    
    Parameters:
    -----------
    x : ndarray
        Audio signal
    
    Returns:
    --------
    features : dict
        Statistical features
    """
    features = {}
    
    # Basic moments
    features['mean'] = float(np.mean(x))
    features['std'] = float(np.std(x))
    features['variance'] = float(np.var(x))
    
    # Absolute value statistics
    abs_x = np.abs(x)
    features['abs_mean'] = float(np.mean(abs_x))
    features['abs_std'] = float(np.std(abs_x))
    
    # Median absolute deviation (robust measure of dispersion)
    features['mad'] = float(np.median(np.abs(x - np.median(x))))
    
    # Range statistics
    features['range'] = float(np.max(x) - np.min(x))
    features['abs_range'] = float(np.max(abs_x) - np.min(abs_x))
    
    # Count-based features
    threshold = 0.1 * np.max(abs_x)
    features['count_above_threshold'] = int(np.sum(abs_x > threshold))
    features['ratio_above_threshold'] = features['count_above_threshold'] / len(x)
    
    return features

## 8. Master Feature Extraction Function

This function orchestrates all feature extraction, creating a comprehensive feature vector for each audio recording.

In [ ]:
def extract_all_features(x: np.ndarray, sr: int, 
                        record_name: str = "",
                        depth_mm: float = None) -> Dict[str, float]:
    """
    Extract comprehensive feature set from audio signal.
    
    This master function calls all individual feature extraction functions
    and combines results into a single feature dictionary.
    
    Parameters:
    -----------
    x : ndarray
        Audio signal
    sr : int
        Sample rate
    record_name : str
        Recording identifier
    depth_mm : float
        Cutting depth in mm (ground truth)
    
    Returns:
    --------
    features : dict
        Complete feature dictionary
    """
    features = {
        'record_name': record_name,
        'depth_mm': depth_mm if depth_mm is not None else np.nan,
        'duration_s': len(x) / sr,
        'sample_rate': sr,
    }
    
    print(f"Extracting features for {record_name}...")
    
    # Time-domain features
    print("  - Time-domain features...")
    features.update(compute_time_domain_features(x, sr))
    
    # Frequency-domain features
    print("  - Frequency-domain features...")
    features.update(compute_frequency_domain_features(x, sr))
    
    # Band power features
    print("  - Band power features...")
    features.update(compute_band_power_features(x, sr))
    
    # Machining-specific features
    print("  - Machining-specific features...")
    features.update(compute_machining_specific_features(x, sr))
    
    # Time-frequency features
    print("  - Time-frequency features...")
    features.update(compute_timefrequency_features(x, sr))
    
    # Statistical features
    print("  - Statistical features...")
    features.update(compute_statistical_features(x))
    
    print(f"  -> Extracted {len(features)} features total\n")
    
    return features

## 9. Feature Extraction and DataFrame Creation

Now we extract features from all recordings and organize them into a pandas DataFrame for analysis.

In [ ]:
# Extract features from all recordings
all_features = []

for record in records:
    features = extract_all_features(
        x=record['seg'],
        sr=record['sr'],
        record_name=record['name'],
        depth_mm=record['depth_mm']
    )
    all_features.append(features)

# Create DataFrame
df_features = pd.DataFrame(all_features)

# Sort by depth for easier viewing
df_features = df_features.sort_values('depth_mm').reset_index(drop=True)

print(f"\nFeature extraction complete!")
print(f"Total features: {len(df_features.columns)}")
print(f"Total recordings: {len(df_features)}")
print(f"\nDataFrame shape: {df_features.shape}")

In [ ]:
# Display first few rows
print("\nFirst few columns of feature data:")
display(df_features.iloc[:, :20].head())

# Show feature categories
print("\n" + "="*60)
print("Feature Categories:")
print("="*60)

time_features = [c for c in df_features.columns if any(x in c.lower() for x in 
                 ['rms', 'peak', 'crest', 'kurtosis', 'skewness', 'zcr', 'envelope', 'waveform'])]
freq_features = [c for c in df_features.columns if any(x in c.lower() for x in 
                 ['spectral', 'freq', 'rolloff', 'entropy'])]
band_features = [c for c in df_features.columns if 'band' in c.lower()]
machining_features = [c for c in df_features.columns if any(x in c.lower() for x in 
                      ['chatter', 'force', 'roughness', 'acoustic_emission', 'modulation'])]
timefreq_features = [c for c in df_features.columns if any(x in c.lower() for x in 
                     ['flux', 'variation', 'centroid_modulation', 'bandwidth_modulation'])]

print(f"\nTime-domain: {len(time_features)} features")
print(f"Frequency-domain: {len(freq_features)} features")
print(f"Band power: {len(band_features)} features")
print(f"Machining-specific: {len(machining_features)} features")
print(f"Time-frequency: {len(timefreq_features)} features")

## 10. Feature Analysis and Correlation with Depth

Let's analyze which features show the strongest correlation with cutting depth, as these will be most useful for depth prediction and validation.

In [ ]:
# Select only numeric features for correlation analysis
numeric_features = df_features.select_dtypes(include=[np.number]).columns.tolist()

# Remove metadata columns
exclude_cols = ['depth_mm', 'duration_s', 'sample_rate']
feature_cols = [c for c in numeric_features if c not in exclude_cols]

# Compute correlation with depth
correlations = []
for col in feature_cols:
    corr = df_features[['depth_mm', col]].corr().iloc[0, 1]
    correlations.append({
        'feature': col,
        'correlation': corr,
        'abs_correlation': abs(corr)
    })

df_corr = pd.DataFrame(correlations).sort_values('abs_correlation', ascending=False)

# Display top correlated features
print("\n" + "="*60)
print("TOP 20 FEATURES CORRELATED WITH CUTTING DEPTH")
print("="*60)
print(df_corr.head(20).to_string(index=False))

# Categorize top features
print("\n" + "="*60)
print("TOP FEATURES BY CATEGORY")
print("="*60)

top_n = 30
top_features = df_corr.head(top_n)['feature'].tolist()

for category, features in [
    ('Time-domain', time_features),
    ('Frequency-domain', freq_features),
    ('Band power', band_features),
    ('Machining-specific', machining_features),
    ('Time-frequency', timefreq_features)
]:
    category_top = [f for f in top_features if f in features]
    if category_top:
        print(f"\n{category}:")
        for f in category_top[:5]:  # Show top 5 per category
            corr = df_corr[df_corr['feature'] == f]['correlation'].values[0]
            print(f"  {f:40s} {corr:+.3f}")

## 11. Visualization: Feature Trends vs. Depth

Visualize how key features vary with cutting depth.

In [ ]:
# Plot top correlated features
top_plot_features = df_corr.head(12)['feature'].tolist()

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, feature in enumerate(top_plot_features):
    ax = axes[idx]
    
    # Plot feature vs depth
    ax.plot(df_features['depth_mm'], df_features[feature], 'o-', 
           markersize=8, linewidth=2)
    
    # Get correlation
    corr = df_corr[df_corr['feature'] == feature]['correlation'].values[0]
    
    ax.set_xlabel('Cutting Depth (mm)', fontsize=10)
    ax.set_ylabel(feature.replace('_', ' ').title(), fontsize=9)
    ax.set_title(f'{feature}\nr = {corr:.3f}', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('feature_extraction/feature_depth_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved as 'feature_depth_correlation.png'")

## 12. Feature Importance and Selection

Let's identify the most informative features using variance and mutual information.

In [ ]:
# Compute coefficient of variation for each feature
cv_scores = []

for col in feature_cols:
    mean_val = df_features[col].mean()
    std_val = df_features[col].std()
    cv = std_val / (abs(mean_val) + 1e-10)
    
    cv_scores.append({
        'feature': col,
        'cv': cv,
        'mean': mean_val,
        'std': std_val
    })

df_cv = pd.DataFrame(cv_scores).sort_values('cv', ascending=False)

print("\n" + "="*60)
print("FEATURES WITH HIGHEST VARIANCE (Coefficient of Variation)")
print("="*60)
print("\nHigh CV indicates features that vary significantly across depths")
print(df_cv.head(20).to_string(index=False))

## 13. Save Features to CSV

Export the complete feature set for further analysis or machine learning.

In [ ]:
# Save to CSV
output_path = 'feature_extraction/audio_features_complete.csv'
df_features.to_csv(output_path, index=False)

print(f"\nFeatures saved to: {output_path}")
print(f"Shape: {df_features.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_features.columns, 1):
    print(f"{i:3d}. {col}")

# Also save correlation analysis
corr_path = 'feature_extraction/feature_correlations_with_depth.csv'
df_corr.to_csv(corr_path, index=False)
print(f"\nCorrelation analysis saved to: {corr_path}")

## 14. Feature Summary Statistics

Generate summary statistics grouped by cutting depth.

In [ ]:
# Summary by depth
print("\n" + "="*60)
print("SUMMARY STATISTICS BY CUTTING DEPTH")
print("="*60)

# Select key features for summary
key_features = df_corr.head(10)['feature'].tolist()
summary_cols = ['depth_mm'] + key_features

summary = df_features[summary_cols].set_index('depth_mm')
display(summary.T)

# Compute percentage change from minimum to maximum depth
print("\n" + "="*60)
print("PERCENTAGE CHANGE (Min to Max Depth)")
print("="*60)

min_depth_idx = df_features['depth_mm'].idxmin()
max_depth_idx = df_features['depth_mm'].idxmax()

for feature in key_features[:10]:
    val_min = df_features.loc[min_depth_idx, feature]
    val_max = df_features.loc[max_depth_idx, feature]
    pct_change = ((val_max - val_min) / (abs(val_min) + 1e-10)) * 100
    
    print(f"{feature:40s} {pct_change:+8.1f}%")

## 15. Conclusions and Recommendations

### Feature Set Summary

This notebook extracted **~150+ features** from acoustic emission signals in micro-milling, organized into six categories:

1. **Time-Domain Features** (~25 features): Capture amplitude characteristics, signal shape, and temporal structure
2. **Frequency-Domain Features** (~30 features): Reveal spectral content, harmonic structure, and frequency distribution
3. **Band Power Features** (~30 features): Energy distribution across machining-relevant frequency bands
4. **Machining-Specific Features** (~20 features): Tool chatter, cutting forces, surface roughness proxies
5. **Time-Frequency Features** (~25 features): Spectral dynamics and modulation characteristics
6. **Statistical Features** (~15 features): General statistical characterizations

### Key Findings

The correlation analysis reveals which features are most sensitive to cutting depth variations. Features showing strong correlations (|r| > 0.8) are prime candidates for:
- Depth prediction models
- Process monitoring
- Quality control

### Recommended Next Steps

1. **Feature Selection**: Use the correlation analysis to select a subset of ~20-30 most informative features
2. **Machine Learning**: Build regression models to predict depth from features
3. **Cross-Validation**: Validate feature stability across different cutting conditions
4. **Physics Validation**: Verify that high-correlation features align with machining physics
5. **Real-Time Implementation**: Consider computational cost for online monitoring

### Literature-Grounded Approach

All features are based on established research in:
- Acoustic emission in manufacturing
- Tool condition monitoring
- Surface quality prediction
- Vibration analysis in machining

This ensures the feature set is not just mathematically comprehensive but also physically meaningful and interpretable in the context of micro-milling operations.